# Cell 1 - Data Loading & Grain Validation

## Objective

Load the core datasets required for the E-Commerce SLA Analysis project and verify their structure before performing any cleaning or analysis.

### Why This Step Matters

Before merging tables, it is important to understand:

* What each table represents
* The level of detail (grain) of each table
* Whether key identifiers are unique

### Dataset Grain Overview

| Table | Grain | Primary Key |
|---------|---------|---------|
| Orders | One row per order | `order_id` |
| Order Items | One row per product within an order | (`order_id`, `order_item_id`) |
| Customers | One row per customer | `customer_id` |
| Sellers | One row per seller | `seller_id` |
| Products | One row per product | `product_id` |

> **Analyst Note:** Understanding table grain is essential before performing joins. Merging tables with different grains without aggregation can lead to row duplication and incorrect metrics.

### Validation Checks

In this section we:

1. Load all required datasets.
2. Check the shape of each table.
3. Verify that order_id is unique in the Orders table.
4. Compare row counts and unique order counts to understand where duplication naturally exists.

Analyst Note: Never merge tables until you understand their grain. Many data quality issues come from joining tables with different levels of detail.

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# Load all CSV files
# Grain note for each table (verify against shape printed below):
#   orders     -> one row per order
#   items      -> one row per order LINE ITEM (finer grain than orders)
#   sellers    -> one row per seller
#   customers  -> one row per customer (1:1 with orders in this dataset)
#   products   -> one row per product

orders   = pd.read_csv('../data/raw/olist_orders_dataset.csv')
items    = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
sellers  = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
customers= pd.read_csv('../data/raw/olist_customers_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')

print(f'Orders:    {orders.shape}')
print(f'Items:     {items.shape}')
print(f'Sellers:   {sellers.shape}')
print(f'Customers: {customers.shape}')
print(f'Products:  {products.shape}')

# Grain validation: confirm orders has one row per order_id
print(f"\norders rows: {len(orders):,} | unique order_id: {orders['order_id'].nunique():,}")
print(f"items rows:  {len(items):,} | unique order_id: {items['order_id'].nunique():,}")

Orders:    (99441, 8)
Items:     (112650, 7)
Sellers:   (3095, 4)
Customers: (99441, 5)
Products:  (32951, 9)

orders rows: 99,441 | unique order_id: 99,441
items rows:  112,650 | unique order_id: 98,666


### Understanding the Output

- **Orders:** 99,441 rows and 99,441 unique `order_id` values → one row per order.
- **Items:** 112,650 rows but only 98,666 unique `order_id` values → some orders contain multiple products.

### Cell 2 - Why Aggregate the Items Table?

Directly merging Orders and Items would duplicate orders with multiple products.

To maintain **one row per order**, we aggregate:

- `sum(price)` → Total order value
- `sum(freight_value)` → Total shipping cost
- `count(order_item_id)` → Number of products

In [32]:
# How many orders have items from MORE than one seller?
sellers_per_order = items.groupby('order_id')['seller_id'].nunique()

print("Distribution of distinct sellers per order:")
print(sellers_per_order.value_counts().sort_index())

print(f"\nOrders with multiple sellers: {(sellers_per_order > 1).sum():,}")
print(f"Total orders with items: {len(sellers_per_order):,}")
print(f"Percentage multi-seller: {(sellers_per_order > 1).mean()*100:.2f}%")

Distribution of distinct sellers per order:
seller_id
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64

Orders with multiple sellers: 1,278
Total orders with items: 98,666
Percentage multi-seller: 1.30%


### Seller Aggregation Decision

- Only **1.30%** of orders involve multiple sellers (1,278 of 98,666).
- We use `first()` for `seller_id` to maintain **one row per order**.
- Create an `is_multi_seller` flag to identify these orders.

> **Analyst Note:** Multi-seller orders are rare (1.30%), so `first()` is acceptable for order-level metrics. These orders will be excluded from seller-level performance analysis to avoid attribution bias.

### Cell 3 - Aggregate Order Items to Order Level

The Items table contains one row per product within an order. To match the grain of the Orders table, we aggregate item-level data into order-level metrics.

We create:
- `num_items` → Number of products in the order
- `total_price` → Total order value
- `total_freight` → Total shipping cost
- `seller_id` → Representative seller (`first()`)
- `is_multi_seller` → Flag for orders with multiple sellers

> Goal: Create a dataset with one row per order before merging with the Orders table.

In [38]:
# Collapse items (one row per line item) down to one row per order.
# Grain change: 112,650 item rows -> ~98,666 order rows (one per order_id)
#
# Aggregation choices and why:
#   - price, freight_value -> sum(): total order value/freight across all items
#   - seller_id            -> first(): see note below
#   - num_items            -> count(): how many line items this order had
#   - is_multi_seller flag -> nunique() > 1: 1.30% of orders, ~1,278 total
#
# Note on seller_id: 1.30% of orders involve >1 seller. We take the first
# seller for order-level metrics (SLA breach is an order-level outcome).
# For seller-level rankings (Day 2, Query 6), we will exclude
# is_multi_seller == 1 to avoid misattributing performance to the wrong seller.

items_agg = items.groupby('order_id').agg(
    num_items=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    seller_id=('seller_id', 'first'),
    n_distinct_sellers=('seller_id', 'nunique')
).reset_index()

items_agg['is_multi_seller'] = (items_agg['n_distinct_sellers'] > 1).astype(int)

print(f"items_agg shape: {items_agg.shape}")
print(f"Expected ~98,666 rows (one per order with items): {'PASS' if items_agg.shape[0] == 98666 else 'CHECK'}")
print(f"\nMulti-seller orders flagged: {items_agg['is_multi_seller'].sum():,}")
print(items_agg.head())

items_agg shape: (98666, 7)
Expected ~98,666 rows (one per order with items): PASS

Multi-seller orders flagged: 1,278
                           order_id  num_items  total_price  total_freight  \
0  00010242fe8c5a6d1ba2dd792cb16214          1        58.90          13.29   
1  00018f77f2f0320c557190d7a144bdd3          1       239.90          19.93   
2  000229ec398224ef6ca0657da4fc703e          1       199.00          17.87   
3  00024acbcdf0a6daa1e931b038114c75          1        12.99          12.79   
4  00042b26cf59d7ce69dfabb4e55b4fd9          1       199.90          18.14   

                          seller_id  n_distinct_sellers  is_multi_seller  
0  48436dade18ac8b2bce089ec2a041202                   1                0  
1  dd7ddc04e1b6c2c614352b383efe2d36                   1                0  
2  5b51032eddd242adc84c38acab88f23d                   1                0  
3  9d7a1d34a5052409006425275ba1c2b4                   1                0  
4  df560393f3a51e74553ab94004ba5c87  

## Cell 4 — Investigating Orders Without Item Rows & Merging Tables

### What we are doing
Checking the 775 orders that have no rows in `order_items`, then merging
`orders`, `items_agg`, `sellers`, and `customers` into one master dataframe.

### Why this order matters
Before merging, we need to understand *why* those 775 orders have no item rows.
If they are all cancelled or unavailable, they will be removed anyway when we
filter to `order_status = 'delivered'` in Cell 5 — meaning the merge join type
(`left` vs `inner`) makes no practical difference for our analysis.
If any are `delivered`, we need to keep them for SLA metrics but exclude them
from seller-level analysis where `seller_id` is required.

### Grain after this merge
One row per `order_id` — the merge must not change this.
We verify `len(df) == len(orders)` immediately after the left join to confirm
no fan-out occurred.

### Potential pitfalls
- Using `inner` join without checking would silently drop those 775 orders
- Not verifying row count after merge hides accidental fan-outs
- Joining `sellers` before aggregating `items` would have caused the fan-out
  problem we discussed (now avoided because `items_agg` is already one row per order)
  

In [43]:
# What are the 775 orders with no item rows?
# Before deciding to drop or keep, check their status and timestamps
orders_without_items = orders[~orders['order_id'].isin(items['order_id'])]

print(f"Orders with no item rows: {len(orders_without_items):,}")
print(f"\nStatus breakdown:")
print(orders_without_items['order_status'].value_counts())
print(f"\nSample rows — do they have valid delivery timestamps?")
print(orders_without_items[[
    'order_id', 'order_status',
    'order_purchase_timestamp',
    'order_estimated_delivery_date',
    'order_delivered_customer_date'
]].head(10).to_string())

Orders with no item rows: 775

Status breakdown:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

Sample rows — do they have valid delivery timestamps?
                              order_id order_status order_purchase_timestamp order_estimated_delivery_date order_delivered_customer_date
266   8e24261a7e58791d10cb1bf9da94df5c  unavailable      2017-11-16 15:09:28           2017-12-05 00:00:00                           NaN
586   c272bcd21c287498b4883c7512019702  unavailable      2018-01-31 11:31:37           2018-02-16 00:00:00                           NaN
687   37553832a3a89c9b2db59701c357ca67  unavailable      2017-08-14 17:38:02           2017-09-05 00:00:00                           NaN
737   d57e15fb07fd180f06ab3926b39edcd2  unavailable      2018-01-08 19:39:03           2018-02-06 00:00:00                           NaN
1130  00b1cb0320190ca0daa2c88b35206009     canceled      2018-08-28 15:26:39 

## Cell 5 — Merging All Tables Into One Master Dataframe

### What we are doing
Merging `orders`, `items_agg`, `sellers`, and `customers` into a single master
dataframe using left joins on their respective keys. We then immediately validate
that the grain (one row per `order_id`) has not changed.

### Why left join, not inner join
The 775 orders with no item rows are all non-delivered (cancelled/unavailable).
A left join preserves them for now — they will be removed naturally in Cell 6
when we filter to `order_status = 'delivered'`. This keeps the removal reason
clean: *business logic*, not join mechanics.

### Grain validation
After every merge we assert `len(df) == len(orders)`.
If this fails, a fan-out has occurred and must be investigated before proceeding.

### Join sequence and why this order
1. `orders` → `items_agg` on `order_id` — safe because `items_agg` is already
   one row per order (aggregated in Cell 2)
2. result → `sellers` on `seller_id` — safe because each order row already has
   exactly one `seller_id` after step 1
3. result → `customers` on `customer_id` — 1:1 relationship confirmed earlier

In [47]:
# Merge orders + items_agg + sellers + customers into one master dataframe.
#
# Join type: LEFT join throughout — preserves all 99,441 orders (grain stays intact).
# The 775 orders with no item rows will get NaN for seller/price columns.
# These are all non-delivered (cancelled/unavailable) and will be removed
# in Cell 6 when we filter to order_status = 'delivered'.
# This is the correct place to remove them — by business reason, not by join type.
#
# Join sequence:
#   orders -> items_agg  : on order_id  (1:1 after aggregation)
#   result -> sellers    : on seller_id (many orders per seller — no fan-out risk
#                          because seller_id is already one value per order row)
#   result -> customers  : on customer_id (1:1 with orders in this dataset)

df = orders.merge(items_agg, on='order_id', how='left')
df = df.merge(sellers,   on='seller_id',   how='left')
df = df.merge(customers, on='customer_id', how='left')

# Grain validation — must still be 99,441 rows after all merges
assert len(df) == len(orders), (
    f"Fan-out detected: expected {len(orders):,} rows, got {len(df):,}"
)

print(f"Merge complete — shape: {df.shape}")
print(f"Grain check PASSED: {len(df):,} rows == {len(orders):,} original orders")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nNaN check on key columns after merge:")
print(df[['seller_id','total_price','seller_state','customer_state']].isnull().sum())

Merge complete — shape: (99441, 21)
Grain check PASSED: 99,441 rows == 99,441 original orders

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'num_items', 'total_price', 'total_freight', 'seller_id', 'n_distinct_sellers', 'is_multi_seller', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

NaN check on key columns after merge:
seller_id         775
total_price       775
seller_state      775
customer_state      0
dtype: int64


## Cell 6 — Converting Date Columns to Datetime

### What we are doing
Converting all 5 date-related columns from raw object/string type to proper
pandas datetime, so we can do arithmetic on them (subtracting dates to get
day differences) in later cells.

### Why this is its own cell, separate from filtering and SLA logic
Type conversion is a distinct concern from business logic. Keeping it isolated
means if a date parsing issue arises, we can debug it without touching the
SLA calculation logic, and the notebook reads as one clear step at a time.

### Potential pitfalls
- pd.to_datetime() will silently fail to parse malformed dates as NaT unless
  errors='raise' is set — we check for new NaNs introduced by conversion
- Comparing across rows before conversion (e.g. accidentally subtracting strings)
  would error loudly, but it's worth confirming dtype explicitly rather than assuming

In [50]:
# Convert all 5 timestamp columns to proper datetime.
# Compare NaN count before and after — should be identical (conversion adds
# no new missing values; it only changes representation).

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

nan_before = df[date_cols].isnull().sum()

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

nan_after = df[date_cols].isnull().sum()

print("Dtypes after conversion:")
print(df[date_cols].dtypes)

print("\nNaN count comparison (before vs after — should match):")
comparison = pd.DataFrame({'before': nan_before, 'after': nan_after})
print(comparison)

assert (nan_before == nan_after).all(), "Conversion introduced new NaNs — investigate"
print("\nNo new NaNs introduced — conversion is clean.")

Dtypes after conversion:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

NaN count comparison (before vs after — should match):
                               before  after
order_purchase_timestamp            0      0
order_approved_at                  14     14
order_delivered_carrier_date        1      1
order_delivered_customer_date       0      0
order_estimated_delivery_date       0      0

No new NaNs introduced — conversion is clean.


## Cell 7 — Filtering to Delivered Orders Only

### What we are doing
Filtering the dataset to only `order_status == 'delivered'`. This is the
business-logic point where the 775 orders with no item rows (all
cancelled/unavailable, confirmed in Cell 4) are removed.

### Why filter here, not earlier
SLA analysis only makes sense for orders that actually completed delivery.
A cancelled order has no `order_delivered_customer_date`, so it can't have
a meaningful "early/late" outcome. Filtering by status (a business reason)
rather than by "has a seller_id" (a structural side-effect) keeps the logic
honest and easy to explain later.

### Validation
After filtering, we expect:
  - Row count drops by roughly (99,441 - 96,478) ≈ 2,963 non-delivered orders
    (775 with no items + other delivered-adjacent statuses like shipped,
    processing, etc. that exist among orders WITH item rows too)
  - order_delivered_customer_date should have zero NaNs after this filter
    (every delivered order must have a delivery date)

### Potential pitfalls
- Assuming only the 775 orders get removed — there are likely other
  non-delivered orders among the ones that DO have item rows
  (e.g. shipped-but-not-yet-delivered). We check the actual drop count below.
- Not re-checking NaN counts after filtering would let a bad assumption slide through silently
- Data quality finding - 8 orders have status='delivered' but no recorded delivery date. These are
dropped explicitly with their own validation step, since SLA calculation is
impossible without a delivery timestamp. This is 0.008% of delivered orders —
immaterial to overall conclusions, but documented rather than silently absorbed.

In [52]:
# Filter to delivered orders only.
before = len(df)
status_counts_before = df['order_status'].value_counts()

df = df[df['order_status'] == 'delivered'].copy()

after_status_filter = len(df)

print(f"Rows before filter: {before:,}")
print(f"Rows after status filter: {after_status_filter:,}")
print(f"Rows removed by status filter: {before - after_status_filter:,}")

# Data quality check: some orders marked 'delivered' have no delivery date.
# This is a source data inconsistency, not something our pipeline caused.
missing_delivery_date = df['order_delivered_customer_date'].isnull().sum()
print(f"\nOrders marked 'delivered' but missing delivery date: {missing_delivery_date}")
print(f"As % of delivered orders: {missing_delivery_date / after_status_filter * 100:.3f}%")

# Drop these explicitly — we cannot compute SLA outcomes without a delivery date,
# and 8 rows out of 96,478 (0.008%) is immaterial to overall findings.
df = df[df['order_delivered_customer_date'].notnull()].copy()

after = len(df)
print(f"\nRows after dropping missing-delivery-date orders: {after:,}")
print(f"Total rows removed in Cell 7: {before - after:,}")

# Validation: every remaining row must have a real delivery date
remaining_nan_delivery = df['order_delivered_customer_date'].isnull().sum()
assert remaining_nan_delivery == 0, "Still found NaN delivery dates — investigate further"
print("\nValidation passed — every remaining row has a real delivery date.")

Rows before filter: 96,470
Rows after status filter: 96,470
Rows removed by status filter: 0

Orders marked 'delivered' but missing delivery date: 0
As % of delivered orders: 0.000%

Rows after dropping missing-delivery-date orders: 96,470
Total rows removed in Cell 7: 0

Validation passed — every remaining row has a real delivery date.


## Cell 8 — SLA Feature Engineering

### What we are doing
Creating 7 new columns that make SLA analysis possible — converting raw
timestamps into actionable metrics: delivery promise vs reality, breach
classification, and the seller vs carrier time split.

### Why this is the most important cell in the project
Every insight, chart, and SQL query downstream depends on these columns.
Getting the arithmetic wrong here — even by one sign — silently corrupts
every metric that follows.

### The key business metric: seller_processing_days
Time from order placed to courier collection — entirely the seller's
responsibility. A courier like DPD can't recover time lost before the
parcel reaches the depot. If seller processing accounts for 60%+ of total
delivery time, the platform's biggest lever is not faster couriers —
it's faster sellers.

### Potential pitfalls
- `days_vs_sla` must be (delivered - estimated), not the reverse —
  positive = late, negative = early. A sign error flips every insight.
- Date arithmetic returns timedelta objects — call `.dt.days` to get
  integers, otherwise comparisons and aggregations silently fail.
- NaT propagation: any NaN in a timestamp column produces NaT in the
  result. We validate no new NaNs were introduced after engineering.

In [60]:
# Engineer 7 SLA columns. All arithmetic is (later_date - earlier_date).dt.days
# Positive days_vs_sla = late. Negative = early.

# 1. Days promised to customer (purchase → estimated delivery)
df['promised_days'] = (
    df['order_estimated_delivery_date'] - df['order_purchase_timestamp']
).dt.days

# 2. Actual days experienced by customer (purchase → actual delivery)
df['actual_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

# 3. Days vs SLA — core metric (negative = early, positive = late)
df['days_vs_sla'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days

# 4. SLA breach binary flag (1 = late, 0 = on time or early)
df['sla_breached'] = (df['days_vs_sla'] > 0).astype(int)

# 5. Seller processing time (purchase → courier collection)
#    Entirely seller's responsibility — time before courier touches the parcel
df['seller_processing_days'] = (
    df['order_delivered_carrier_date'] - df['order_purchase_timestamp']
).dt.days

# 6. Carrier delivery time (courier collection → customer delivery)
#    Courier's responsibility — what DPD actually controls
df['carrier_delivery_days'] = (
    df['order_delivered_customer_date'] - df['order_delivered_carrier_date']
).dt.days

# 7. Breach severity classification
def classify_breach(days):
    if pd.isna(days) or days <= 0: return 'On Time / Early'
    elif days <= 3:                 return 'Minor (1-3 days late)'
    elif days <= 7:                 return 'Moderate (4-7 days late)'
    else:                           return 'Severe (7+ days late)'

df['breach_severity'] = df['days_vs_sla'].apply(classify_breach)

# ── Validation ──────────────────────────────────────────────────────────────
print("=== SLA Feature Engineering Complete ===")
print(f"\nNew columns added: promised_days, actual_days, days_vs_sla, "
      f"sla_breached, seller_processing_days, carrier_delivery_days, breach_severity")

# Check no NaNs introduced in numeric SLA columns
sla_cols = ['promised_days','actual_days','days_vs_sla',
            'sla_breached','seller_processing_days','carrier_delivery_days']
print(f"\nNaN check on SLA columns:")
print(df[sla_cols].isnull().sum())

# Key headline metrics
print(f"\n=== Headline Metrics ===")
print(f"Total delivered orders:    {len(df):,}")
print(f"SLA breached orders:       {df['sla_breached'].sum():,}")
print(f"Overall breach rate:       {df['sla_breached'].mean()*100:.1f}%")
print(f"Avg days vs SLA:           {df['days_vs_sla'].mean():.1f} days")
print(f"Avg seller processing:     {df['seller_processing_days'].mean():.1f} days")
print(f"Avg carrier delivery:      {df['carrier_delivery_days'].mean():.1f} days")

print(f"\nBreach severity breakdown:")
print(df['breach_severity'].value_counts())

# Seller vs carrier split
total_avg = df['actual_days'].mean()
seller_avg = df['seller_processing_days'].mean()
carrier_avg = df['carrier_delivery_days'].mean()
print(f"\nDelivery time split:")
print(f"  Seller processing: {seller_avg:.1f} days ({seller_avg/total_avg*100:.0f}% of total)")
print(f"  Carrier delivery:  {carrier_avg:.1f} days ({carrier_avg/total_avg*100:.0f}% of total)")

# 1 NaN found in seller_processing_days and carrier_delivery_days
# Root cause: one order has a missing order_delivered_carrier_date
# Impact: immaterial (1 out of 96,470 = 0.001%)
# Decision: leave in dataset — this order still has valid sla_breached,
# days_vs_sla, and breach_severity. Only seller/carrier split analysis
# will exclude it naturally via dropna() when needed.
nan_carrier = df['order_delivered_carrier_date'].isnull().sum()
print(f"\nOrders missing carrier date: {nan_carrier} ({nan_carrier/len(df)*100:.3f}%)")
print("Decision: retained — sla_breached and days_vs_sla are still valid for these rows.")

=== SLA Feature Engineering Complete ===

New columns added: promised_days, actual_days, days_vs_sla, sla_breached, seller_processing_days, carrier_delivery_days, breach_severity

NaN check on SLA columns:
promised_days             0
actual_days               0
days_vs_sla               0
sla_breached              0
seller_processing_days    1
carrier_delivery_days     1
dtype: int64

=== Headline Metrics ===
Total delivered orders:    96,470
SLA breached orders:       6,534
Overall breach rate:       6.8%
Avg days vs SLA:           -11.9 days
Avg seller processing:     2.7 days
Avg carrier delivery:      8.9 days

Breach severity breakdown:
breach_severity
On Time / Early             89936
Severe (7+ days late)        2862
Minor (1-3 days late)        1870
Moderate (4-7 days late)     1802
Name: count, dtype: int64

Delivery time split:
  Seller processing: 2.7 days (23% of total)
  Carrier delivery:  8.9 days (73% of total)

Orders missing carrier date: 1 (0.001%)
Decision: retained 

## Cell 9 — Saving the Clean Dataset

### What we are doing
Saving the fully cleaned and engineered dataframe to `data/clean/orders_clean.csv`.
This is the single source of truth for all downstream work — Day 2 SQL analysis,
Day 3 charts, and Day 4 Power BI all load from this file.

### Why save here and not earlier
We only save once every transformation and validation has passed. Saving
intermediate or unvalidated data risks downstream analysis being built on
a broken foundation without anyone noticing.

### What this file contains
- 96,470 delivered orders (down from 99,441 raw)
- 28 columns including 7 engineered SLA features
- One row per order — grain confirmed and maintained throughout

In [61]:
# Save the clean, engineered dataset to data/clean/
# This file is the single source of truth for all downstream analysis.

output_path = '../data/clean/orders_clean.csv'
df.to_csv(output_path, index=False)

# Verify the save was successful by reading it back and checking shape
df_verify = pd.read_csv(output_path)

print("=== Day 1 Complete ===")
print(f"\nFile saved to: {output_path}")
print(f"Shape written:   {df.shape}")
print(f"Shape read back: {df_verify.shape}")

assert df.shape == df_verify.shape, "Shape mismatch — file did not save correctly"
print("\nSave verified — shapes match.")

print(f"\n=== Final Dataset Summary ===")
print(f"Total delivered orders:     {len(df):,}")
print(f"Total columns:              {len(df.columns)}")
print(f"SLA breach rate:            {df['sla_breached'].mean()*100:.1f}%")
print(f"Avg days early/late:        {df['days_vs_sla'].mean():.1f} days")
print(f"Avg seller processing:      {df['seller_processing_days'].mean():.1f} days")
print(f"Avg carrier delivery:       {df['carrier_delivery_days'].mean():.1f} days")
print(f"\nColumns saved:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col}")

=== Day 1 Complete ===

File saved to: ../data/clean/orders_clean.csv
Shape written:   (96470, 28)
Shape read back: (96470, 28)

Save verified — shapes match.

=== Final Dataset Summary ===
Total delivered orders:     96,470
Total columns:              28
SLA breach rate:            6.8%
Avg days early/late:        -11.9 days
Avg seller processing:      2.7 days
Avg carrier delivery:       8.9 days

Columns saved:
   1. order_id
   2. customer_id
   3. order_status
   4. order_purchase_timestamp
   5. order_approved_at
   6. order_delivered_carrier_date
   7. order_delivered_customer_date
   8. order_estimated_delivery_date
   9. num_items
  10. total_price
  11. total_freight
  12. seller_id
  13. n_distinct_sellers
  14. is_multi_seller
  15. seller_zip_code_prefix
  16. seller_city
  17. seller_state
  18. customer_unique_id
  19. customer_zip_code_prefix
  20. customer_city
  21. customer_state
  22. promised_days
  23. actual_days
  24. days_vs_sla
  25. sla_breached
  26. seller_